[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BSLJunhyeonJeon/AI_COP/blob/main/session4/notebooks/01_train_lab.ipynb)

# session4 · 01 · 학습(Training) — 하이퍼파라미터 실험실

- **이 노트북에서 배우는 것**: 3회차는 학습을 **돌려 봤다**. 4회차는 학습이 **잘 되고 있는지 읽는 법**을 배운다. 손잡이(학습률·에폭·증강·데이터 양)를 돌리면 성능이 어떻게 변하는지 눈으로 본다.
- **입력**: 없음 — beans 잎 병해 데이터(공개, MIT)를 코드로 내려받습니다.
- **출력**: 실험 그림 11장 + 실험 기록 `outputs/runs.json`

> **런타임 > 런타임 유형 변경 > GPU(T4)** 로 먼저 설정하세요.
> **실험 1회가 30~60초**입니다. 이 노트북은 "누르고 기다리는" 노트북이 아니라 **여러 번 돌려 보는** 노트북입니다.
> 모든 실험 결과는 `outputs/runs.json` 에 쌓입니다. 런타임이 끊겨도 뒤 셀이 앞 결과를 읽어 그림을 다시 그립니다.
> ⚠️ **실행 순서**: `01` → `03_build_html` 을 먼저 끝내고, **`02_pose_demo` 는 맨 마지막**에 실행하세요(`02` 가 mediapipe 를 설치해 런타임 패키지 구성을 바꿀 수 있습니다).
> 그림 안 글자는 폰트 호환을 위해 영문입니다.

In [ ]:
# 셀 1 · 환경 감지 + 프로젝트 루트 확보 (session2·3 과 동일 패턴 — 분기는 이 셀 한 곳)
import os, subprocess

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

SESSION = "session4"
REPO_URL = "https://github.com/BSLJunhyeonJeon/AI_COP"
REPO_DIR = "/content/AI_COP"
SESSION_DIR = REPO_DIR + "/" + SESSION


def acquire_project():
    if os.path.isdir(REPO_DIR):
        print("이미 존재:", REPO_DIR, "(재클론 건너뜀)")
        try:
            r = subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)
            if r.returncode != 0:
                print("  (git pull 실패 — 기존 캐시 버전 사용)")
        except Exception as e:
            print("  (git pull 건너뜀:", e, ")")
    else:
        print("레포 클론:", REPO_URL, "->", REPO_DIR)
        try:
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
        except Exception as e:
            print("clone 실패(네트워크/권한 확인):", e)
    return SESSION_DIR if os.path.isdir(SESSION_DIR) else None


def find_root_local(marker="requirements.txt"):
    start = os.path.abspath(os.getcwd())
    d = start
    while True:
        if os.path.exists(os.path.join(d, marker)):
            return d
        parent = os.path.dirname(d)
        if parent == d:
            print("[주의] '" + marker + "' 를 못 찾음. 현재 폴더를 루트로 가정:", start)
            return start
        d = parent


PROJECT_ROOT = acquire_project() if IN_COLAB else find_root_local()
if not (PROJECT_ROOT and os.path.isdir(PROJECT_ROOT)):
    raise RuntimeError(
        "세션 루트를 확보하지 못했습니다. "
        "코랩이면 레포 클론 실패이니 네트워크 확인 후 이 셀(셀 1)을 다시 ▶ 실행하세요. "
        "로컬이면 session4/ 안에서 노트북을 열었는지 확인하세요."
    )
os.chdir(PROJECT_ROOT)
for d in ("data", "weights", "outputs"):
    os.makedirs(d, exist_ok=True)
print("실행 환경   :", "Colab" if IN_COLAB else "Local")
print("PROJECT_ROOT:", PROJECT_ROOT)

In [ ]:
# 셀 2 · 의존성 설치 + 버전/GPU 확인
# CONVENTIONS 규칙 3 — 여기 출력된 '실제 버전'을 보고 requirements.txt 핀을 확정한다.
import os, sys, subprocess

if not os.path.exists("requirements.txt"):
    raise RuntimeError("requirements.txt 를 찾지 못했습니다. 셀 1을 먼저 ▶ 실행하세요.")
print("requirements.txt 설치 중...")
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=False)
if r.returncode != 0:
    raise RuntimeError("pip install 실패. 위 로그 확인 후 이 셀(셀 2)을 다시 ▶ 실행하세요.")

print("\n[설치된 버전 — requirements.txt 핀 확인용]")
for mod in ["torch", "torchvision", "numpy", "matplotlib", "PIL"]:
    try:
        m = __import__(mod)
        print("  -", ("Pillow" if mod == "PIL" else mod), ":", getattr(m, "__version__", "?"))
    except Exception as e:
        print("  -", mod, ": import 실패 (", e, ")")

# 실험 1회의 소요 시간이 여기서 갈린다 — 학생이 미리 알아야 한다.
import torch
print("\n[GPU]")
if torch.cuda.is_available():
    print("  사용 가능:", torch.cuda.get_device_name(0), " -> 실험 1회 30~60초")
else:
    print("  GPU 없음 -> CPU 로도 돌아가지만 실험 1회가 수 분씩 걸립니다.")
    print("  런타임 > 런타임 유형 변경 > GPU(T4) 로 바꾸시길 권합니다.")

In [ ]:
# 셀 3 · beans 데이터 확보 (다운로드 -> 해제 -> 장수 검증 -> 샘플 그림)
# 출처: Makerere AI Lab ibean / HuggingFace AI-Lab-Makerere/beans (MIT)
# zip 안에 이미 split 폴더가 들어 있어 풀면 그대로 ImageFolder 구조가 된다. datasets 라이브러리 불필요.
import os, zipfile, urllib.request, collections
import matplotlib.pyplot as plt
from PIL import Image

BASE_URL = "https://huggingface.co/datasets/AI-Lab-Makerere/beans/resolve/main/data/"
DATA = os.path.join("data", "beans")
SPLITS = ["train", "validation", "test"]
CLASSES = ["angular_leaf_spot", "bean_rust", "healthy"]   # ImageFolder 알파벳 순 = 0,1,2
IMG_EXT = (".jpg", ".jpeg", ".png")

# ------------------------------------------------------------------
# 실측 장수 (zip 을 직접 열어 센 값. 추측 아님)
#   train.zip 안에는 파일이 1035개 있지만 그중 1개는 이미지가 아니다:
#   'healthy_train.120tore' — 헤더가 b'\x00\x00\x00\x01Bud1' 인 macOS 파인더 메타데이터(.DS_Store)다.
#   torchvision ImageFolder 는 이미지 확장자만 읽으므로 이 파일을 세지 않는다.
#   -> 파일 수 1035 / 실제 이미지 1034 (healthy 341). 아래 셀에서 그 1개를 지우고 1034 로 맞춘다.
# ------------------------------------------------------------------
EXPECTED = {
    "train":      {"angular_leaf_spot": 345, "bean_rust": 348, "healthy": 341},
    "validation": {"angular_leaf_spot": 44,  "bean_rust": 45,  "healthy": 44},
    "test":       {"angular_leaf_spot": 43,  "bean_rust": 43,  "healthy": 42},
}


def count_images(d):
    if not os.path.isdir(d):
        return 0
    return len([f for f in os.listdir(d) if f.lower().endswith(IMG_EXT)])


def download(url, path):
    """HF 는 UA 없는 요청을 막을 수 있어 헤더를 붙인다. .part 로 받고 완료 후 이름을 바꾼다(중단 안전)."""
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req) as resp, open(path + ".part", "wb") as f:
        got = 0
        while True:
            chunk = resp.read(1 << 20)
            if not chunk:
                break
            f.write(chunk)
            got += len(chunk)
        print("    받음 %.1f MB" % (got / 1e6))
    os.replace(path + ".part", path)


os.makedirs(DATA, exist_ok=True)
for s in SPLITS:
    if sum(count_images(os.path.join(DATA, s, c)) for c in CLASSES) > 0:
        print("이미 있음:", os.path.join(DATA, s), "(다운로드·해제 건너뜀)")
        continue
    zpath = os.path.join("data", s + ".zip")
    if not os.path.exists(zpath):
        print("다운로드:", BASE_URL + s + ".zip")
        try:
            download(BASE_URL + s + ".zip", zpath)
        except Exception as e:
            raise RuntimeError("다운로드 실패(%s): %s — 네트워크 확인 후 이 셀(셀 3)을 다시 ▶ 실행하세요." % (s, e))
    print("해제:", zpath, "->", DATA)
    with zipfile.ZipFile(zpath) as zf:
        zf.extractall(DATA)

# 이미지가 아닌 파일 제거 (위 주석의 macOS 메타데이터 1개)
junk = []
for s in SPLITS:
    for c in CLASSES:
        d = os.path.join(DATA, s, c)
        if not os.path.isdir(d):
            continue
        for fn in sorted(os.listdir(d)):
            p = os.path.join(d, fn)
            if os.path.isfile(p) and not fn.lower().endswith(IMG_EXT):
                os.remove(p)
                junk.append("%s/%s/%s" % (s, c, fn))
if junk:
    print("\n[정리] 이미지가 아닌 파일 %d개 제거:" % len(junk), junk)
    print("       (배포된 zip 에 섞여 들어간 macOS 메타데이터입니다. 이미지가 아니라서 학습에 쓰지 않습니다.)")

# --- split x 클래스 장수 표 ---
print("\n%-12s %8s %20s %12s %10s" % ("split", "total", CLASSES[0], CLASSES[1], CLASSES[2]))
print("-" * 66)
counts, mismatch = {}, []
for s in SPLITS:
    per = {c: count_images(os.path.join(DATA, s, c)) for c in CLASSES}
    counts[s] = per
    print("%-12s %8d %20d %12d %10d" % (s, sum(per.values()), per[CLASSES[0]], per[CLASSES[1]], per[CLASSES[2]]))
    for c in CLASSES:
        if per[c] != EXPECTED[s][c]:
            mismatch.append("%s/%s: 실제 %d != 기대 %d" % (s, c, per[c], EXPECTED[s][c]))
print("-" * 66)
if mismatch:
    print("[경고] 기대 장수와 다릅니다 ->", mismatch)
    print("       data/beans 를 지우고 셀 3을 다시 ▶ 실행하면 새로 받습니다.")
else:
    print("[확인] 모든 split x 클래스 장수가 기대값과 일치합니다.")

n_tr, n_va, n_te = (sum(counts[s].values()) for s in SPLITS)

# --- 샘플 3장 (클래스당 1장) ---
fig, axes = plt.subplots(1, 3, figsize=(9.6, 3.6))
for ax, c in zip(axes, CLASSES):
    d = os.path.join(DATA, "train", c)
    files = sorted(f for f in os.listdir(d) if f.lower().endswith(IMG_EXT))
    im = Image.open(os.path.join(d, files[0]))
    ax.imshow(im)
    ax.set_title("%s\n%dx%d" % (c, im.size[0], im.size[1]), fontsize=11)
    ax.axis("off")
fig.suptitle("beans dataset — 3 classes (bean leaf disease)")
plt.tight_layout()
plt.savefig(os.path.join("outputs", "04_dataset.png"), dpi=130)
plt.show()
print("저장: outputs/04_dataset.png")

# 화면 숫자와 멘트가 어긋나지 않게, 코드가 직접 센 값을 그대로 넣는다.
ratio = [counts["train"][c] / max(1, min(counts["train"].values())) for c in CLASSES]
print()
print("train %d / val %d / test %d — 세 클래스가 거의 균형입니다(%.2f : %.2f : %.2f). "
      "3회차 BCCD 는 11배 불균형이었죠. 오늘은 데이터 탓을 할 수 없습니다."
      % (n_tr, n_va, n_te, ratio[0], ratio[1], ratio[2]))

In [ ]:
# 셀 4 · 실험 함수 정의 (이 셀은 '정의만' 합니다 — 학습은 셀 5부터)
import os, json, time, random, copy
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torchvision.models import resnet18, ResNet18_Weights

DATA = os.path.join("data", "beans")
RUNS_JSON = os.path.join("outputs", "runs.json")
CLASSES = ["angular_leaf_spot", "bean_rust", "healthy"]
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]   # ImageNet 정규화
BATCH = 32
# 스펙 기본값은 2. 윈도우 로컬 주피터는 워커 프로세스 spawn 이 자주 멈춰서 0으로 내린다(코랩은 2).
NUM_WORKERS = 0 if os.name == "nt" else 2
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# val/test 변환은 절대 증강하지 않는다 — 시험지를 흔들면 점수를 믿을 수 없다.
EVAL_TF = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(224),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD),
])


def make_train_tf(aug):
    ops = [transforms.Resize(256), transforms.CenterCrop(224)]
    if aug:
        ops += [transforms.RandomHorizontalFlip(),
                transforms.RandomRotation(15),
                transforms.ColorJitter(0.3, 0.3, 0.3)]
    ops += [transforms.ToTensor(), transforms.Normalize(MEAN, STD)]
    return transforms.Compose(ops)


def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def load_runs():
    """outputs/runs.json 을 읽는다. 없거나 깨졌으면 빈 dict."""
    if os.path.exists(RUNS_JSON):
        try:
            with open(RUNS_JSON, encoding="utf-8") as f:
                return json.load(f)
        except Exception as e:
            print("[주의] runs.json 을 읽지 못해 새로 시작합니다:", e)
    return {}


def save_run(rec):
    """실험 결과를 runs.json 에 누적한다(같은 이름이면 덮어씀)."""
    runs = load_runs()
    runs[rec["name"]] = rec
    os.makedirs("outputs", exist_ok=True)
    with open(RUNS_JSON, "w", encoding="utf-8") as f:
        json.dump(runs, f, ensure_ascii=False, indent=1)
    return runs


def get_run(name):
    return load_runs().get(name)


def subset_indices(targets, n_per_class, seed):
    """클래스당 n_per_class 장을 시드 고정으로 뽑는다. n_per_class=None 이면 전체."""
    if n_per_class is None:
        return list(range(len(targets)))
    by = {}
    for i, t in enumerate(targets):
        by.setdefault(int(t), []).append(i)
    rng = random.Random(seed)
    idx = []
    for c in sorted(by):
        pool = by[c]
        idx += pool if len(pool) <= n_per_class else rng.sample(pool, n_per_class)
    idx.sort()
    return idx


def build_model():
    """ImageNet 사전학습 ResNet18 의 마지막 층만 3클래스로 갈아끼운다(전체 파인튜닝, freeze 없음)."""
    m = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
    m.fc = nn.Linear(512, len(CLASSES))
    return m.to(DEVICE)


def make_loaders(aug, n_per_class, seed):
    train_full = ImageFolder(os.path.join(DATA, "train"), transform=make_train_tf(aug))
    val_ds = ImageFolder(os.path.join(DATA, "validation"), transform=EVAL_TF)   # 133장 전체
    idx = subset_indices(train_full.targets, n_per_class, seed)
    g = torch.Generator(); g.manual_seed(seed)
    train_loader = DataLoader(Subset(train_full, idx), batch_size=BATCH, shuffle=True,
                              num_workers=NUM_WORKERS, generator=g, drop_last=False)
    val_loader = DataLoader(val_ds, batch_size=BATCH, shuffle=False, num_workers=NUM_WORKERS)
    return train_loader, val_loader, len(idx), len(val_ds)


@torch.no_grad()
def evaluate(model, loader, crit):
    model.eval()
    tot_loss, correct, n = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        out = model(x)
        tot_loss += float(crit(out, y)) * y.size(0)
        correct += int((out.argmax(1) == y).sum())
        n += y.size(0)
    return tot_loss / max(1, n), correct / max(1, n)


def run_experiment(name, lr, epochs, aug=False, n_per_class=100, seed=42):
    """실험 1회 = 학습 + 매 epoch 검증. 결과를 outputs/runs.json 에 저장하고 dict 로 돌려준다.

    최고 val_acc 시점의 가중치는 weights/<name>_best.pt 로 저장한다
    (셀 8·9 가 다시 학습하지 않고 그 모델을 그대로 쓰기 위해).
    """
    set_seed(seed)
    train_loader, val_loader, n_train, n_val = make_loaders(aug, n_per_class, seed)
    model = build_model()
    crit = nn.CrossEntropyLoss()
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9)   # 스케줄러 없음(학습률 효과를 순수하게 보려고)

    print("[%s] lr=%g epochs=%d aug=%s train=%d장 val=%d장 device=%s"
          % (name, lr, epochs, aug, n_train, n_val, DEVICE.type))
    hist = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best_acc, best_epoch, best_state = -1.0, -1, None
    t0 = time.time()
    for ep in range(1, epochs + 1):
        model.train()
        tot_loss, correct, n = 0.0, 0, 0
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            out = model(x)
            loss = crit(out, y)
            loss.backward()
            opt.step()
            tot_loss += float(loss) * y.size(0)
            correct += int((out.argmax(1) == y).sum())
            n += y.size(0)
        tr_loss, tr_acc = tot_loss / max(1, n), correct / max(1, n)
        va_loss, va_acc = evaluate(model, val_loader, crit)
        hist["train_loss"].append(tr_loss); hist["train_acc"].append(tr_acc)
        hist["val_loss"].append(va_loss); hist["val_acc"].append(va_acc)
        is_best = va_acc > best_acc
        if is_best:
            best_acc, best_epoch = va_acc, ep
            best_state = copy.deepcopy({k: v.detach().cpu() for k, v in model.state_dict().items()})
        # 초보자가 '뭔가 돌아가는 것'을 봐야 한다 — epoch 마다 한 줄.
        print("  epoch %2d/%d | train loss %.4f acc %.3f | val loss %.4f acc %.3f%s"
              % (ep, epochs, tr_loss, tr_acc, va_loss, va_acc, "   <- best" if is_best else ""))
    seconds = time.time() - t0

    os.makedirs("weights", exist_ok=True)
    wpath = os.path.join("weights", "%s_best.pt" % name)
    if best_state is not None:
        torch.save(best_state, wpath)

    rec = {"name": name, "lr": lr, "epochs": epochs, "aug": bool(aug),
           "n_per_class": n_per_class, "n_train": n_train, "seed": seed,
           "best_val_acc": best_acc, "best_epoch": best_epoch,
           "seconds": seconds, "weights": wpath, "history": hist}
    save_run(rec)
    print("  -> 최고 val_acc %.3f (epoch %d) | %.1f초 | 가중치 %s | runs.json 저장"
          % (best_acc, best_epoch, seconds, wpath))
    return rec


print("정의 완료: run_experiment(name, lr, epochs, aug=False, n_per_class=100, seed=42)")
print("장치:", DEVICE, "| 배치:", BATCH, "| num_workers:", NUM_WORKERS)
print("기록 파일:", RUNS_JSON, "(실험할 때마다 누적, 같은 이름은 덮어씀)")

In [ ]:
# 셀 5 · 실험 A: 학습률(보폭) — 5개 값을 순차 실행 (예상 2.5분 안팎)
import os, time
import matplotlib.pyplot as plt

if "run_experiment" not in globals():
    raise RuntimeError("셀 4(실험 함수 정의)를 먼저 ▶ 실행하세요.")

LR_LIST = [(1e-1, "1e-1"), (1e-2, "1e-2"), (1e-3, "1e-3"), (1e-4, "1e-4"), (1e-6, "1e-6")]
EPOCHS_A = 8

t_all = time.time()
for lr, tag in LR_LIST:
    run_experiment("A_lr_%s" % tag, lr=lr, epochs=EPOCHS_A, aug=False, n_per_class=100)
print("\n실험 A 전체 소요: %.1f초" % (time.time() - t_all))

runs = load_runs()

# --- (1) 겹쳐 그린 비교 그림 ---
fig, ax = plt.subplots(figsize=(7.6, 5.0))
for lr, tag in LR_LIST:
    r = runs.get("A_lr_%s" % tag)
    if not r:
        continue
    va = r["history"]["val_acc"]
    ax.plot(range(1, len(va) + 1), va, marker="o", markersize=3, label="lr=%s" % tag)
ax.set_xlabel("epoch"); ax.set_ylabel("val accuracy")
ax.set_xlim(0.5, EPOCHS_A + 0.5); ax.set_ylim(0.0, 1.0)
ax.grid(alpha=0.25)
ax.set_title("Experiment A — learning rate (same data, same model, same time)")
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig(os.path.join("outputs", "04_lr_compare.png"), dpi=130)
plt.show()
print("저장: outputs/04_lr_compare.png")

# --- (2) HTML 슬라이더용 개별 5장 ---
# 슬라이더에서 갈아끼울 때 튀지 않도록 축 범위·크기·여백·DPI 를 하드코딩해 5장 모두 동일하게 만든다.
FIGSIZE, DPI = (6.4, 4.4), 130
ADJUST = dict(left=0.12, right=0.97, top=0.90, bottom=0.13)
for lr, tag in LR_LIST:
    r = runs.get("A_lr_%s" % tag)
    if not r:
        print("  건너뜀(기록 없음): lr =", tag)
        continue
    h = r["history"]
    ep = range(1, len(h["val_acc"]) + 1)
    fig, ax = plt.subplots(figsize=FIGSIZE)
    fig.subplots_adjust(**ADJUST)                 # tight_layout / bbox_inches 쓰지 않음(크기 동일 보장)
    ax.plot(ep, h["train_acc"], marker="o", markersize=3, label="train acc")
    ax.plot(ep, h["val_acc"], marker="s", markersize=3, label="val acc")
    ax.set_xlim(0.5, EPOCHS_A + 0.5); ax.set_ylim(0.0, 1.0)
    ax.set_xlabel("epoch"); ax.set_ylabel("accuracy")
    ax.grid(alpha=0.25)
    ax.set_title("lr = %s   |   best val acc = %.3f" % (tag, r["best_val_acc"]))
    ax.legend(loc="lower right")
    out = os.path.join("outputs", "04_lr_%s.png" % tag)
    plt.savefig(out, dpi=DPI)
    plt.close(fig)
    print("  저장:", out, " (best val acc %.3f)" % r["best_val_acc"])

best = max((runs["A_lr_%s" % t] for _, t in LR_LIST if "A_lr_%s" % t in runs),
           key=lambda r: r["best_val_acc"])
print()
print("가장 좋았던 학습률: %g (best val acc %.3f)" % (best["lr"], best["best_val_acc"]))
print("같은 데이터, 같은 모델, 같은 시간. 바뀐 건 숫자 하나(학습률)뿐입니다.")

In [ ]:
# 셀 6 · 실험 B: 과적합(에폭) — 셀 5에서 가장 좋았던 lr 로 30 epoch
import os
import matplotlib.pyplot as plt

if "run_experiment" not in globals():
    raise RuntimeError("셀 4(실험 함수 정의)를 먼저 ▶ 실행하세요.")

runs = load_runs()
a_runs = [r for k, r in runs.items() if k.startswith("A_lr_")]
if not a_runs:
    raise RuntimeError("실험 A 기록이 없습니다. 셀 5를 먼저 ▶ 실행하세요.")
BEST_LR = max(a_runs, key=lambda r: r["best_val_acc"])["lr"]
print("셀 5에서 가장 좋았던 학습률:", BEST_LR)

NAME_B, EPOCHS_B = "B_overfit", 30
rec = run_experiment(NAME_B, lr=BEST_LR, epochs=EPOCHS_B, aug=False, n_per_class=100)

h = rec["history"]
ep = list(range(1, len(h["val_acc"]) + 1))
fig, ax = plt.subplots(figsize=(7.6, 5.0))
ax.plot(ep, h["train_acc"], label="train acc")
ax.plot(ep, h["val_acc"], label="val acc")
# train 이 val 을 앞선 구간 = 외우기 시작한 구간
ax.fill_between(ep, h["val_acc"], h["train_acc"],
                where=[t >= v for t, v in zip(h["train_acc"], h["val_acc"])],
                alpha=0.15, color="tab:red", label="train-val gap (memorizing)")
be = rec["best_epoch"]
ax.axvline(be, color="k", linestyle="--", linewidth=1.2)
ax.annotate("should have stopped here\n(epoch %d, val acc %.3f)" % (be, rec["best_val_acc"]),
            xy=(be, rec["best_val_acc"]), xytext=(0.30, 0.28), textcoords="axes fraction",
            arrowprops=dict(arrowstyle="->", color="k"), fontsize=10)
ax.set_xlim(0.5, EPOCHS_B + 0.5); ax.set_ylim(0.0, 1.05)
ax.set_xlabel("epoch"); ax.set_ylabel("accuracy"); ax.grid(alpha=0.25)
ax.set_title("Experiment B — overfitting (lr=%g, %d images, no augmentation)" % (BEST_LR, rec["n_train"]))
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig(os.path.join("outputs", "04_overfit.png"), dpi=130)
plt.show()
print("저장: outputs/04_overfit.png")

wasted = rec["epochs"] - rec["best_epoch"]
print()
print("최종 train_acc : %.3f" % h["train_acc"][-1])
print("최종 val_acc   : %.3f" % h["val_acc"][-1])
print("최고 val_acc   : %.3f (epoch %d)" % (rec["best_val_acc"], rec["best_epoch"]))
print("헛돈 epoch     : %d 번 (%d epoch 중 %d 이후로는 검증 점수가 더 나아지지 않았습니다)"
      % (wasted, rec["epochs"], rec["best_epoch"]))
print("train 은 계속 올라가는데 val 이 멈추면 — 배우는 게 아니라 외우는 중입니다.")

In [ ]:
# 셀 6.5 · 실험 D: 데이터 양 (클래스당 100장 vs train 전체) — 선생님 승인으로 추가된 실험
# 짝 비교의 상대는 셀 5의 'A_lr_<best>' 입니다(같은 lr·같은 epoch·증강 없음, 다른 건 데이터 양뿐).
import os
import matplotlib.pyplot as plt

if "run_experiment" not in globals():
    raise RuntimeError("셀 4(실험 함수 정의)를 먼저 ▶ 실행하세요.")

runs = load_runs()
a_runs = [r for k, r in runs.items() if k.startswith("A_lr_")]
if not a_runs:
    raise RuntimeError("실험 A 기록이 없습니다. 셀 5를 먼저 ▶ 실행하세요.")
small = max(a_runs, key=lambda r: r["best_val_acc"])       # 100장/클래스, 8 epoch
BEST_LR, EPOCHS_D = small["lr"], small["epochs"]

big = run_experiment("D_data_full", lr=BEST_LR, epochs=EPOCHS_D, aug=False, n_per_class=None)

fig, ax = plt.subplots(figsize=(7.6, 5.0))
for r, lab in [(small, "100 per class (%d imgs)" % small["n_train"]),
               (big, "full train set (%d imgs)" % big["n_train"])]:
    va = r["history"]["val_acc"]
    ax.plot(range(1, len(va) + 1), va, marker="o", markersize=3, label=lab)
ax.set_xlim(0.5, EPOCHS_D + 0.5); ax.set_ylim(0.0, 1.0)
ax.set_xlabel("epoch"); ax.set_ylabel("val accuracy"); ax.grid(alpha=0.25)
ax.set_title("Experiment D — how much data? (lr=%g, %d epochs, no augmentation)" % (BEST_LR, EPOCHS_D))
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig(os.path.join("outputs", "04_datasize.png"), dpi=130)
plt.show()
print("저장: outputs/04_datasize.png")

d = big["best_val_acc"] - small["best_val_acc"]
print()
print("100장/클래스 (%d장): 최고 val_acc %.3f | %.1f초" % (small["n_train"], small["best_val_acc"], small["seconds"]))
print("train 전체  (%d장): 최고 val_acc %.3f | %.1f초" % (big["n_train"], big["best_val_acc"], big["seconds"]))
print("차이: %+.3f  (데이터는 %.1f배, 시간은 %.1f배)"
      % (d, big["n_train"] / max(1, small["n_train"]), big["seconds"] / max(1e-6, small["seconds"])))
print("데이터를 %.1f배 늘리는 값이 이 차이만큼 하는지 — 판단은 결과를 보고 하세요."
      % (big["n_train"] / max(1, small["n_train"])))

In [ ]:
# 셀 7 · 실험 C: 증강 — 셀 6과 완전히 같은 설정에서 aug=True 만 바꾼다
import os
import matplotlib.pyplot as plt

if "run_experiment" not in globals():
    raise RuntimeError("셀 4(실험 함수 정의)를 먼저 ▶ 실행하세요.")

runs = load_runs()
base = runs.get("B_overfit")
if not base:
    raise RuntimeError("셀 6(실험 B) 기록이 없습니다. 셀 6을 먼저 ▶ 실행하세요.")

NAME_C = "C_aug"
rec = run_experiment(NAME_C, lr=base["lr"], epochs=base["epochs"], aug=True,
                     n_per_class=base["n_per_class"])

# 셀 6 결과는 runs.json 에서 읽는다 — 다시 학습하지 않는다.
fig, ax = plt.subplots(figsize=(7.6, 5.0))
for r, lab in [(base, "no augmentation"), (rec, "augmentation (flip + rotate15 + jitter)")]:
    va = r["history"]["val_acc"]
    ax.plot(range(1, len(va) + 1), va, label="%s — best %.3f" % (lab, r["best_val_acc"]))
ax.set_xlim(0.5, base["epochs"] + 0.5); ax.set_ylim(0.0, 1.0)
ax.set_xlabel("epoch"); ax.set_ylabel("val accuracy"); ax.grid(alpha=0.25)
ax.set_title("Experiment C — augmentation on/off (lr=%g, %d imgs, %d epochs)"
             % (base["lr"], base["n_train"], base["epochs"]))
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig(os.path.join("outputs", "04_aug.png"), dpi=130)
plt.show()
print("저장: outputs/04_aug.png")

d = rec["best_val_acc"] - base["best_val_acc"]
print()
print("증강 없음: 최고 val_acc %.3f (epoch %d)" % (base["best_val_acc"], base["best_epoch"]))
print("증강 있음: 최고 val_acc %.3f (epoch %d)" % (rec["best_val_acc"], rec["best_epoch"]))
print("차이     : %+.3f" % d)
# 결과를 좋게 보이게 고르지 않는다 — 실제 수치대로 말한다.
if d > 0.01:
    print("=> 증강이 도움이 됐습니다. 같은 사진을 뒤집고 돌려 '새로운 사진처럼' 보여준 효과입니다.")
elif d < -0.01:
    print("=> 증강이 오히려 나빴습니다. 예상과 다르죠? 3회차 RBC 처럼, 예상과 다른 결과도 그대로 수업 소재입니다.")
    print("   증강은 공짜가 아닙니다 — 문제를 어렵게 만들어 같은 epoch 안에 덜 배울 수도 있습니다.")
else:
    print("=> 사실상 차이가 없습니다. '증강하면 좋아진다'는 항상 참인 법칙이 아닙니다.")
print("val/test 는 절대 증강하지 않았습니다 — 시험지를 흔들면 점수를 믿을 수 없습니다.")

In [ ]:
# 셀 8 · 지표 심화 — 정확도 한 숫자 뒤에 뭐가 있는지 (혼동행렬 / precision·recall·F1 / 트레이드오프)
# sklearn 을 쓰지 않고 numpy 로 직접 계산합니다(코랩 사전설치에 기대지 않고, 계산식을 눈으로 보려고).
import os
import numpy as np
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder

if "build_model" not in globals():
    raise RuntimeError("셀 4(실험 함수 정의)를 먼저 ▶ 실행하세요.")

runs = load_runs()
rec = runs.get("C_aug")
if not rec:
    raise RuntimeError("셀 7(실험 C, 증강) 기록이 없습니다. 셀 7을 먼저 ▶ 실행하세요.")

# 가중치 확보 순서 — 런타임이 끊겨도 이 셀만 다시 돌릴 수 있게 3단 폴백.
CANDS = [rec.get("weights"),                                   # 1) 셀 7이 저장한 최고 epoch 가중치
         os.path.join("weights", "beans_resnet18_aug_backup.pt")]  # 2) 레포 커밋 백업(라이브 보험)
wpath = next((c for c in CANDS if c and os.path.exists(c)), None)
model = build_model()
if wpath:
    if wpath.endswith("backup.pt"):
        print("[백업] 커밋된 백업 가중치로 진행합니다:", wpath)
    model.load_state_dict(torch.load(wpath, map_location=DEVICE))
else:
    print("[복구] 가중치 파일이 없어 runs.json 의 설정으로 실험 C를 다시 돌립니다(30~60초)...")
    rec = run_experiment("C_aug", lr=rec["lr"], epochs=rec["epochs"], aug=True,
                         n_per_class=rec["n_per_class"], seed=rec["seed"])
    model.load_state_dict(torch.load(rec["weights"], map_location=DEVICE))
model.eval()

# validation 133장 전체로 예측 (증강 없음)
val_ds = ImageFolder(os.path.join(DATA, "validation"), transform=EVAL_TF)
loader = DataLoader(val_ds, batch_size=BATCH, shuffle=False, num_workers=NUM_WORKERS)
probs_all, y_all = [], []
with torch.no_grad():
    for x, y in loader:
        p = torch.softmax(model(x.to(DEVICE)), dim=1)
        probs_all.append(p.cpu().numpy()); y_all.append(y.numpy())
probs = np.concatenate(probs_all); y_true = np.concatenate(y_all)
y_pred = probs.argmax(1)
K = len(CLASSES)
print("검증 %d장 예측 완료 | 전체 정확도 %.3f" % (len(y_true), float((y_pred == y_true).mean())))

# --- 1) 혼동행렬 3x3 (numpy 로 직접) ---
cm = np.zeros((K, K), dtype=int)
for t, p in zip(y_true, y_pred):
    cm[t, p] += 1

SHORT = ["angular", "rust", "healthy"]
fig, ax = plt.subplots(figsize=(6.6, 5.4))
ax.imshow(cm, cmap="Blues")
for i in range(K):
    row = max(1, cm[i].sum())
    for j in range(K):
        ax.text(j, i, "%d\n(%.0f%%)" % (cm[i, j], 100.0 * cm[i, j] / row),
                ha="center", va="center",
                color="white" if cm[i, j] > cm.max() * 0.55 else "black", fontsize=12)
ax.set_xticks(range(K)); ax.set_xticklabels(SHORT)
ax.set_yticks(range(K)); ax.set_yticklabels(SHORT)
ax.set_xlabel("predicted"); ax.set_ylabel("true (% = share of the true row)")
ax.set_title("Confusion matrix — validation (%d images)" % len(y_true))
plt.tight_layout()
plt.savefig(os.path.join("outputs", "04_confusion.png"), dpi=130)
plt.show()
print("저장: outputs/04_confusion.png")

# --- 2) 클래스별 precision / recall / F1 (numpy 로 직접) ---
print()
print("%-20s %8s %8s %8s %8s" % ("class", "support", "precis.", "recall", "F1"))
print("-" * 56)
for i, c in enumerate(CLASSES):
    tp = int(cm[i, i])
    fp = int(cm[:, i].sum() - tp)     # 다른 클래스인데 i 라고 부른 것
    fn = int(cm[i, :].sum() - tp)     # i 인데 다른 클래스라고 부른 것
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec_ = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * prec * rec_ / (prec + rec_) if (prec + rec_) else 0.0
    print("%-20s %8d %8.3f %8.3f %8.3f" % (c, int(cm[i].sum()), prec, rec_, f1))
print("-" * 56)
print("precision = 이 이름을 붙인 것 중 진짜 비율 | recall = 진짜 중 찾아낸 비율")

# --- 3) precision-recall 트레이드오프 (healthy vs 병(2클래스 병합)) ---
# 양성 = '병'. 점수 = P(angular) + P(rust). 임계값을 올릴수록 확신 있는 것만 '병'이라 부른다.
score = probs[:, 0] + probs[:, 1]
is_disease = (y_true != CLASSES.index("healthy"))
ths = np.arange(0.10, 0.901, 0.05)
P, R = [], []
for t in ths:
    pred_pos = score >= t
    tp = int((pred_pos & is_disease).sum())
    fp = int((pred_pos & ~is_disease).sum())
    fn = int((~pred_pos & is_disease).sum())
    P.append(tp / (tp + fp) if (tp + fp) else 1.0)
    R.append(tp / (tp + fn) if (tp + fn) else 0.0)

fig, ax = plt.subplots(figsize=(7.6, 5.0))
ax.plot(ths, P, marker="o", markersize=3, label="precision (fewer false alarms)")
ax.plot(ths, R, marker="s", markersize=3, label="recall (fewer missed diseases)")
ax.set_xlim(0.05, 0.95); ax.set_ylim(0.0, 1.02)
ax.set_xlabel("threshold on P(disease)"); ax.set_ylabel("score"); ax.grid(alpha=0.25)
ax.set_title("Precision-recall trade-off — healthy vs diseased (validation)")
ax.legend(loc="lower left")
plt.tight_layout()
plt.savefig(os.path.join("outputs", "04_tradeoff.png"), dpi=130)
plt.show()
print("저장: outputs/04_tradeoff.png")
print()
print("임계값 0.10 -> precision %.3f / recall %.3f" % (P[0], R[0]))
print("임계값 0.90 -> precision %.3f / recall %.3f" % (P[-1], R[-1]))
print("병든 잎을 놓치지 않으려면 recall, 헛경보를 줄이려면 precision. "
      "어느 쪽이 중요한지는 모델이 아니라 연구 질문이 정합니다.")

In [ ]:
# 셀 9 · 요약표 + test 개봉 (test 는 이 노트북에서 딱 한 번만 씁니다)
import os
import torch
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder

if "build_model" not in globals():
    raise RuntimeError("셀 4(실험 함수 정의)를 먼저 ▶ 실행하세요.")

runs = load_runs()
if not runs:
    raise RuntimeError("outputs/runs.json 이 비어 있습니다. 셀 5~7을 먼저 ▶ 실행하세요.")

order = sorted(runs.values(), key=lambda r: (-r["best_val_acc"], r["name"]))
print("%-14s %8s %7s %6s %8s %10s %8s" % ("name", "lr", "epochs", "aug", "n/class", "best val", "sec"))
print("-" * 70)
for r in order:
    print("%-14s %8g %7d %6s %8s %10.3f %8.1f"
          % (r["name"], r["lr"], r["epochs"], "O" if r["aug"] else "X",
             ("all" if r["n_per_class"] is None else r["n_per_class"]),
             r["best_val_acc"], r["seconds"]))
print("-" * 70)
print("총 실험 %d회 / 총 학습 시간 %.1f초 (%.1f분)"
      % (len(order), sum(r["seconds"] for r in order), sum(r["seconds"] for r in order) / 60.0))

# --- 가장 좋은 설정 하나만 골라 test 개봉 ---
best = order[0]
print("\n선택한 설정:", best["name"], "(val 기준 최고: %.3f)" % best["best_val_acc"])
model = build_model()
wpath = best.get("weights")
if wpath and os.path.exists(wpath):
    model.load_state_dict(torch.load(wpath, map_location=DEVICE))
else:
    print("[복구] 가중치가 없어 runs.json 설정으로 다시 학습합니다...")
    best = run_experiment(best["name"], lr=best["lr"], epochs=best["epochs"], aug=best["aug"],
                          n_per_class=best["n_per_class"], seed=best["seed"])
    model.load_state_dict(torch.load(best["weights"], map_location=DEVICE))
model.eval()

test_ds = ImageFolder(os.path.join(DATA, "test"), transform=EVAL_TF)   # 증강 없음
loader = DataLoader(test_ds, batch_size=BATCH, shuffle=False, num_workers=NUM_WORKERS)
correct = 0
with torch.no_grad():
    for x, y in loader:
        correct += int((model(x.to(DEVICE)).argmax(1).cpu() == y).sum())
test_acc = correct / len(test_ds)
print("test %d장 정확도: %.3f  (val 최고: %.3f)" % (len(test_ds), test_acc, best["best_val_acc"]))
print()
print("방금 test 를 열었습니다. 이제부터 이 결과를 보고 설정을 또 바꾸면, test 는 더 이상 test 가 아닙니다.")

In [ ]:
# 셀 10 · 검증 / 요약
import os

REQUIRED = ["04_dataset.png", "04_lr_compare.png",
            "04_lr_1e-1.png", "04_lr_1e-2.png", "04_lr_1e-3.png", "04_lr_1e-4.png", "04_lr_1e-6.png",
            "04_overfit.png", "04_aug.png", "04_confusion.png", "04_tradeoff.png"]
EXTRA = ["04_datasize.png"]      # 셀 6.5(승인으로 추가된 실험)의 산출물

print("=" * 60)
print(" session4 · 01_train_lab 검증")
print("=" * 60)
ok = 0
for f in REQUIRED:
    p = os.path.join("outputs", f)
    ok += os.path.exists(p)
    print("  %-20s : %s" % (f, "있음" if os.path.exists(p) else "없음"))
print("-" * 60)
for f in EXTRA:
    p = os.path.join("outputs", f)
    print("  %-20s : %s  (추가 실험 셀 6.5)" % (f, "있음" if os.path.exists(p) else "없음"))
rj = os.path.join("outputs", "runs.json")
print("  %-20s : %s" % ("runs.json", "있음" if os.path.exists(rj) else "없음"))
print("-" * 60)
print("  필수 그림: %d / %d" % (ok, len(REQUIRED)))
if ok < len(REQUIRED):
    print("  [주의] 빠진 그림이 있습니다 — 해당 셀(3·5·6·7·8)을 다시 ▶ 실행하세요.")
if os.path.exists(rj):
    try:
        print("  기록된 실험: %d개 -> %s" % (len(load_runs()), ", ".join(sorted(load_runs()))))
    except Exception:
        pass
print("=" * 60)
print("배운 것:")
print(" 1) 학습률은 '보폭'이다 — 너무 크면 넘어지고, 너무 작으면 제자리다. 데이터도 모델도 그대로인데 숫자 하나로 갈린다.")
print(" 2) train 이 오를 때 val 이 멈추면 외우는 중이다 — 오래 돌린다고 좋아지지 않는다. 멈출 지점은 val 이 알려준다.")
print(" 3) 정확도 한 숫자는 요약일 뿐이다 — 혼동행렬·precision·recall 을 봐야 '무엇을 어떻게 틀리는지'가 보인다.")